# testing learn-to-look

One reply from the full S2, on a board with exactly one gold. **New board** deals `n_gold=1` and opening `any`. The settings card can move the agent and the gold; **Render** refuses a board that does not have one gold.

**Generate** builds the training trace and then samples one reply. The trace is `synthetic_prompt` with `forced_note="the gold"`: the same fake history as training (`_fake_rounds`), and a notepad whose value is `the gold` with no region description. The question box is sent verbatim. The default is `look at the target. Are you looking up or down?`

**Load** has two choices. **Full save** is `GemmaS2.load_snapshot` (a directory with `localizer.pt`, `coord_embed.pt`, `train_meta.json`, and either a `gemma/` bundle, an adapter, or `gemma_base`). **Gemma from here, localizer from here** is `GemmaS2.load_parts`: a Gemma directory (`gemma_meta.json` or `adapter_config.json`) plus a separate readout directory. **Refresh** rescans `weights/`.

The picture in the prompt is the clean frame. Each new token is decoded with the previous token's `v` and `v_bar`, the same handoff as the game loop. A click draws that token's **s** in blue, **v** in red, **v_bar** in orange, and the average of **v** and **v_bar** in green. A point outside the unit square is left off the picture; the readout still prints the number. **show all** draws every token at once.

Sampling stops on eos, or at `BEGINNING_MAX_NEW_TOKENS`. **Clear** wipes the log, the tokens, and the dots.

Run this on the GPU box. The cell needs the repo environment (`pygame`, `torch`, `ipywidgets`).


In [ ]:
import html
import io
import json
from pathlib import Path
import math
import os
import random
import sys
import tempfile
import time
import traceback

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import ipywidgets as widgets
import numpy as np
import torch
from IPython.display import HTML, display
from PIL import Image

from agent.game_io import (
    apply_edited_settings_dict,
    game_to_settings_dict,
    new_multi_gold_game,
)
from agent.notebook_ui import settings_editor
from neural_net.loop import _eos_ids, _sample
from neural_net.paths import weights_root
from neural_net.render import canonical_frame
from neural_net.s2 import GemmaS2
from neural_net.s2_pretraining_learn_to_look import (
    BEGINNING_MAX_NEW_TOKENS,
    synthetic_prompt,
)

BOARD_PX = 560
FORCED_NOTE = "the gold"
DEFAULT_QUESTION = "look at the target. Are you looking up or down?"
_RADIUS_ONE = 8
_RADIUS_ALL = 3
_COLOR = {
    "s": (30, 90, 255),
    "v": (255, 0, 0),
    "v_bar": (255, 140, 0),
    "avg": (0, 200, 70),
}
_DRAW_ORDER = ("s", "v_bar", "v", "avg")
_READOUT = ("localizer.pt", "coord_embed.pt", "train_meta.json")

tokens = []
state = {
    "busy": False,
    "ignore_through": 0,
    "gen": 0,
    "s2": None,
    "game": None,
    "frame": None,
    "selected": None,
}


def _label(path):
    root = weights_root()
    try:
        return Path(path).relative_to(root).as_posix()
    except ValueError:
        return Path(path).as_posix()


def _is_readout(path):
    return all((path / name).is_file() for name in _READOUT)


def _is_full(path):
    if not _is_readout(path):
        return False
    if (path / "gemma" / "gemma_meta.json").is_file():
        return True
    if (path / "adapter_config.json").is_file():
        return True
    meta_path = path / "train_meta.json"
    try:
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False
    return bool(meta.get("gemma_base"))


def _scan():
    root = weights_root()
    gemma = []
    localizer = []
    full = []
    if not root.is_dir():
        return gemma, localizer, full
    seen = set()
    for name in ("adapter_config.json", "gemma_meta.json"):
        for marker in root.rglob(name):
            parent = marker.parent
            if parent in seen:
                continue
            seen.add(parent)
            gemma.append(parent)
    for marker in root.rglob("localizer.pt"):
        parent = marker.parent
        if not _is_readout(parent):
            continue
        localizer.append(parent)
        if _is_full(parent):
            full.append(parent)
    gemma.sort(key=lambda path: path.stat().st_mtime, reverse=True)
    localizer.sort(
        key=lambda path: (path / "localizer.pt").stat().st_mtime, reverse=True,
    )
    full.sort(
        key=lambda path: (path / "localizer.pt").stat().st_mtime, reverse=True,
    )
    return gemma, localizer, full


def _options(paths):
    if not paths:
        return [("(none)", "")]
    return [(_label(path), str(path)) for path in paths]


def _keep(dropdown, options):
    selected = dropdown.value
    dropdown.options = options
    values = [value for _, value in options]
    dropdown.value = selected if selected in values else values[0]


def _on_board(x, y):
    return (
        math.isfinite(x) and math.isfinite(y)
        and 0.0 <= x <= 1.0 and 0.0 <= y <= 1.0
    )


def _blit_dot(out, x, y, color, radius):
    if not _on_board(x, y):
        return
    height, width = out.shape[:2]
    col = int(round(float(x) * (width - 1)))
    row = int(round((1.0 - float(y)) * (height - 1)))
    y0 = max(0, row - radius)
    y1 = min(height, row + radius + 1)
    x0 = max(0, col - radius)
    x1 = min(width, col + radius + 1)
    yy, xx = np.ogrid[y0:y1, x0:x1]
    mask = (yy - row) ** 2 + (xx - col) ** 2 <= radius ** 2
    out[y0:y1, x0:x1][mask] = color


def _png(rgb):
    buf = io.BytesIO()
    Image.fromarray(rgb, mode="RGB").save(buf, format="PNG")
    return buf.getvalue()


def _paint_layer(out, rows, radius):
    for key in _DRAW_ORDER:
        color = _COLOR[key]
        for row in rows:
            x, y = row[key]
            _blit_dot(out, x, y, color, radius)


def _overlay(frame):
    selected = state["selected"]
    show = bool(show_all.value) and bool(tokens)
    one = (
        tokens[selected]
        if selected is not None and 0 <= selected < len(tokens)
        else None
    )
    if not show and one is None:
        return frame
    out = np.array(frame, copy=True)
    if show:
        _paint_layer(out, tokens, _RADIUS_ALL)
    if one is not None:
        _paint_layer(out, [one], _RADIUS_ONE)
    return out


def _show_board():
    frame = state["frame"]
    if frame is None:
        return
    board.value = _png(_overlay(frame))


def _visible(piece):
    if piece == "":
        return "∅"
    return piece.replace(" ", "\u00a0")


def _token_markup():
    if not tokens:
        return "<div class='ltl-tokens'>Generate to get clickable tokens.</div>"
    gen = state["gen"]
    selected = state["selected"]
    parts = []
    for row in tokens:
        cls = "ltl-tok"
        if row["eos"]:
            cls += " ltl-eos"
        if row["i"] == selected:
            cls += " ltl-on"
        text = html.escape(_visible(row["piece"]))
        parts.append(
            f"<span class='{cls}' data-gen='{gen}' data-i='{row['i']}'>"
            f"{text}</span>"
        )
    return "<div class='ltl-tokens'>" + "".join(parts) + "</div>"


def _show_tokens():
    token_html.value = _token_markup()


def _point_text(name, xy):
    x, y = xy
    extra = "" if _on_board(x, y) else " (off the board)"
    return f"{name}=({x:.4f}, {y:.4f}){extra}"


def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def _say(text):
    log.append_stdout(text if text.endswith("\n") else text + "\n")


def _set_busy(flag):
    state["busy"] = flag
    for widget in controls:
        widget.disabled = flag
    settings.set_disabled(flag)


def _one_gold(game):
    golds = game_to_settings_dict(game).get("gold") or []
    if len(golds) != 1:
        raise RuntimeError(
            f"this notebook needs exactly one gold, the board has {len(golds)}"
        )
    return (float(golds[0][0]), float(golds[0][1]))


def _fresh_frame():
    state["frame"] = canonical_frame(state["game"])
    _show_board()


def _start_game():
    state["game"] = new_multi_gold_game(n_gold=1, opening="any")
    state["selected"] = None
    tokens.clear()
    state["gen"] += 1
    _fresh_frame()
    _show_tokens()


_FRAME_PATH = os.path.join(tempfile.gettempdir(), "testing_learn_to_look_frame.png")


def _write_frame(frame):
    Image.fromarray(np.ascontiguousarray(frame), mode="RGB").save(_FRAME_PATH)
    return _FRAME_PATH


def _log_prompt(messages):
    for message in messages:
        role = message.get("role", "?")
        content = message.get("content")
        chunks = []
        if isinstance(content, str):
            chunks.append(content)
        elif isinstance(content, list):
            for part in content:
                if not isinstance(part, dict):
                    continue
                if part.get("type") == "image":
                    chunks.append("[image: current frame]")
                elif part.get("type") == "text":
                    chunks.append(part.get("text") or "")
        _say(f"--- {role} ---\n" + "\n".join(chunks))


def _record_token(tokenizer, token_id, coords, eos):
    vals = [float(v) for v in coords.detach().float().cpu().tolist()]
    if len(vals) != 6:
        raise RuntimeError(
            f"coordinate head returned {len(vals)} values, expected 6"
        )
    sx, sy, vx, vy, bx, by = vals
    piece = tokenizer.decode([token_id], skip_special_tokens=False)
    tokens.append({
        "i": len(tokens),
        "id": token_id,
        "piece": piece,
        "eos": token_id in eos,
        "s": (sx, sy),
        "v": (vx, vy),
        "v_bar": (bx, by),
        "avg": ((vx + bx) / 2.0, (vy + by) / 2.0),
    })


def _generate(question_text):
    net = state["s2"]
    net.vl.model.eval()
    gold = _one_gold(state["game"])
    frame = canonical_frame(state["game"])
    state["frame"] = frame
    settings_dict = game_to_settings_dict(state["game"])
    agent = (float(settings_dict["agent_x"]), float(settings_dict["agent_y"]))
    _say(
        f"agent=({agent[0]:.4f}, {agent[1]:.4f})  "
        f"gold=({gold[0]:.4f}, {gold[1]:.4f})"
    )
    built = synthetic_prompt(
        settings_dict,
        random.Random(),
        _write_frame(frame),
        question=question_text,
        forced_note=FORCED_NOTE,
    )
    if built.target_note != FORCED_NOTE:
        raise RuntimeError(
            f"trace note is {built.target_note!r}, expected {FORCED_NOTE!r}"
        )
    _log_prompt(built.messages)
    encoded = net.vl.encode_messages(built.messages)
    prompt_len = int(encoded["input_ids"].shape[1])
    _say(f"prompt tokens {prompt_len}")
    sampling = net.vl._sampling_kwargs()
    eos = _eos_ids(net.vl)
    tokenizer = getattr(net.vl.processor, "tokenizer", net.vl.processor)
    _sync()
    started = time.perf_counter()
    logits, coords, past = net.prefill(encoded)
    seq_len = prompt_len
    for token_index in range(BEGINNING_MAX_NEW_TOKENS):
        token_ids = _sample(logits, sampling)
        token_id = int(token_ids[0].item())
        _record_token(tokenizer, token_id, coords[0], eos)
        if token_id in eos or token_index + 1 == BEGINNING_MAX_NEW_TOKENS:
            break
        logits, coords, past = net.decode(
            token_ids, past, seq_len, prev_coords=coords,
        )
        seq_len += 1
        if token_index % 16 == 0:
            status.value = f"Generating token {token_index + 1}."
    _sync()
    elapsed = time.perf_counter() - started
    ids = [row["id"] for row in tokens]
    text = tokenizer.decode(ids, skip_special_tokens=True).strip()
    if tokens and tokens[-1]["eos"]:
        why = "eos"
    else:
        why = f"cap {BEGINNING_MAX_NEW_TOKENS}"
    _say(f"{len(tokens)} tokens in {elapsed:.1f}s ({why})")
    if text:
        _say(text)
    _say("")


_gemma_paths, _loc_paths, _full_paths = _scan()
_gemma_opts = _options(_gemma_paths)
_loc_opts = _options(_loc_paths)
_full_opts = _options(_full_paths)

load_mode = widgets.RadioButtons(
    options=[
        ("Gemma from here, localizer from here", "parts"),
        ("Full save", "full"),
    ],
    value="full",
    description="Load:",
    style={"description_width": "50px"},
)
gemma_dd = widgets.Dropdown(
    options=_gemma_opts,
    value=_gemma_opts[0][1],
    description="Gemma:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="480px"),
)
loc_dd = widgets.Dropdown(
    options=_loc_opts,
    value=_loc_opts[0][1],
    description="Localizer:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="480px"),
)
full_dd = widgets.Dropdown(
    options=_full_opts,
    value=_full_opts[0][1],
    description="Full:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="640px"),
)
refresh_btn = widgets.Button(description="Refresh")
load_btn = widgets.Button(description="Load", button_style="primary")
new_btn = widgets.Button(description="New board")
clear_btn = widgets.Button(description="Clear")
question = widgets.Textarea(
    value=DEFAULT_QUESTION,
    description="Question:",
    continuous_update=True,
    style={"description_width": "90px"},
    layout=widgets.Layout(width="100%", height="72px"),
)
generate_btn = widgets.Button(description="Generate", button_style="primary")
show_all = widgets.Checkbox(value=False, description="show all", indent=False)
legend = widgets.HTML(
    "<span style='color:#1e5aff'>● s</span> &nbsp; "
    "<span style='color:#ff0000'>● v</span> &nbsp; "
    "<span style='color:#ff8c00'>● v_bar</span> &nbsp; "
    "<span style='color:#00c846'>● average of v and v_bar</span>"
)
readout = widgets.HTML("Click a token after Generate.")
status = widgets.HTML("Load S2, then Generate.")
board = widgets.Image(format="png", width=BOARD_PX, height=BOARD_PX)
token_html = widgets.HTML()
log = widgets.Output(layout=widgets.Layout(
    border="1px solid #ccc",
    height="280px",
    max_height="280px",
    overflow_y="auto",
    width="100%",
    flex="0 0 auto",
))
log.add_class("ltl-log")
pick_box = widgets.Text(value="")
pick_box.add_class("ltl-pick")
pick_box.layout.height = "0px"
pick_box.layout.visibility = "hidden"
pick_box.layout.overflow = "hidden"
settings = settings_editor(height_px=560)
controls = (
    load_mode, gemma_dd, loc_dd, full_dd,
    refresh_btn, load_btn, new_btn, clear_btn,
    question, generate_btn, show_all,
)


def on_refresh(_):
    gemma_paths, loc_paths, full_paths = _scan()
    _keep(gemma_dd, _options(gemma_paths))
    _keep(loc_dd, _options(loc_paths))
    _keep(full_dd, _options(full_paths))
    _say(
        f"{len(gemma_paths)} gemma, {len(loc_paths)} localizer, "
        f"{len(full_paths)} full save(s) under {weights_root()}"
    )


def on_load(_):
    if state["busy"]:
        return
    mode = load_mode.value
    if mode == "parts":
        gemma_path = gemma_dd.value
        loc_path = loc_dd.value
        if not gemma_path or not loc_path:
            _say("pick both a Gemma directory and a localizer directory")
            return
    elif mode == "full":
        full_path = full_dd.value
        if not full_path:
            _say("no full save selected")
            return
    else:
        raise RuntimeError(f"unknown load mode {mode!r}")
    _set_busy(True)
    status.value = "Loading S2."
    try:
        if state["s2"] is None:
            state["s2"] = GemmaS2()
        net = state["s2"]
        if mode == "parts":
            net.load_parts(gemma_path, loc_path)
            loaded = f"gemma {gemma_path}  localizer {loc_path}"
        else:
            net.load_snapshot(full_path)
            loaded = full_path
        n_head = sum(p.numel() for p in net.localizer.parameters())
        device = next(net.vl.model.parameters()).device
        _say(f"loaded {loaded}  localizer {n_head} params  on {device}")
        status.value = "Edit the question if you want, then Generate."
    except Exception:
        status.value = "Load failed."
        _say(traceback.format_exc())
        raise
    finally:
        _set_busy(False)


def on_clear(_):
    if state["busy"]:
        return
    tokens.clear()
    state["selected"] = None
    state["gen"] += 1
    log.clear_output()
    readout.value = "Click a token after Generate."
    _show_tokens()
    _show_board()
    status.value = "Record cleared."


def on_new(_):
    if state["busy"]:
        return
    _start_game()
    view = game_to_settings_dict(state["game"])
    settings.show_view(view)
    gold = _one_gold(state["game"])
    readout.value = "Click a token after Generate."
    _say(
        f"new board  agent=({float(view['agent_x']):.4f}, "
        f"{float(view['agent_y']):.4f})  "
        f"gold=({gold[0]:.4f}, {gold[1]:.4f})  "
        f"openings={len(view.get('openings') or [])}"
    )
    if state["s2"] is None or state["s2"].vl.model is None:
        status.value = "Load S2, then Generate."
    else:
        status.value = "Edit the question if you want, then Generate."


def on_settings_edit(_):
    if state["busy"]:
        return
    settings.enter_edit(game_to_settings_dict(state["game"]))


def on_settings_render(_):
    if state["busy"] or not settings.is_editing():
        return
    try:
        drafted = settings.collect()
        if len(drafted.get("gold") or []) != 1:
            settings.set_error("This notebook needs exactly one gold.")
            return
        state["game"] = apply_edited_settings_dict(drafted)
    except ValueError as exc:
        settings.set_error(f"Error, fix settings before rendering.\n{exc}")
        return
    tokens.clear()
    state["selected"] = None
    state["gen"] += 1
    readout.value = "Click a token after Generate."
    _fresh_frame()
    _show_tokens()
    settings.show_view(game_to_settings_dict(state["game"]))
    _say("rendered settings")


def on_show_all(_change):
    _show_board()


def on_generate(_):
    if state["busy"]:
        return
    if settings.is_editing():
        _say("Render game settings before Generate.")
        return
    if state["s2"] is None or state["s2"].vl.model is None:
        _say("no S2 loaded")
        return
    question_text = question.value
    if not question_text.strip():
        _say("question is empty")
        status.value = "The question box is empty."
        return
    _set_busy(True)
    status.value = "Generating."
    tokens.clear()
    state["selected"] = None
    state["gen"] += 1
    readout.value = "Click a token after Generate."
    _show_tokens()
    _show_board()
    try:
        _generate(question_text)
    except Exception:
        status.value = "Generate failed."
        _say(traceback.format_exc())
        raise
    finally:
        _show_tokens()
        _show_board()
        _set_busy(False)
    status.value = (
        "Click a token. Blue is s, red is v, orange is v_bar, "
        "green is the average of v and v_bar."
    )


def on_pick(change):
    raw = change["new"]
    if not raw or raw.count(",") != 2:
        return
    seq_s, gen_s, idx_s = raw.split(",")
    try:
        seq = int(seq_s)
        gen = int(gen_s)
        index = int(idx_s)
    except ValueError:
        return
    if seq <= state["ignore_through"] or state["busy"]:
        return
    if gen != state["gen"]:
        return
    if index < 0 or index >= len(tokens):
        _say(f"token {index} is not in this reply ({len(tokens)} tokens)")
        return
    state["selected"] = index
    row = tokens[index]
    readout.value = (
        f"token {index} &nbsp; {html.escape(row['piece'])}<br>"
        f"{_point_text('s', row['s'])} &nbsp; "
        f"{_point_text('v', row['v'])}<br>"
        f"{_point_text('v_bar', row['v_bar'])} &nbsp; "
        f"{_point_text('avg', row['avg'])}"
    )
    _show_tokens()
    _show_board()
    _say(
        f"token {index} {row['piece']!r}  "
        f"s={row['s'][0]:.4f},{row['s'][1]:.4f}  "
        f"v={row['v'][0]:.4f},{row['v'][1]:.4f}  "
        f"v_bar={row['v_bar'][0]:.4f},{row['v_bar'][1]:.4f}  "
        f"avg={row['avg'][0]:.4f},{row['avg'][1]:.4f}"
    )
    state["ignore_through"] = seq


refresh_btn.on_click(on_refresh)
load_btn.on_click(on_load)
new_btn.on_click(on_new)
clear_btn.on_click(on_clear)
generate_btn.on_click(on_generate)
show_all.observe(on_show_all, names="value")
settings.edit_btn.on_click(on_settings_edit)
settings.render_btn.on_click(on_settings_render)
pick_box.observe(on_pick, names="value")

_start_game()
settings.show_view(game_to_settings_dict(state["game"]))
_show_tokens()
_say(f"cuda {torch.cuda.is_available()}")
_say(
    f"{len(_gemma_paths)} gemma, {len(_loc_paths)} localizer, "
    f"{len(_full_paths)} full save(s) under {weights_root()}"
)
_say(
    "Generate builds a fake NAMS trace with notepad target: the gold, "
    "then one reply. Click a token to plot s / v / v_bar / their average."
)

display(HTML(
    """
<style>
.ltl-tokens {
  white-space: pre-wrap;
  line-height: 1.8;
  border: 1px solid #ccc;
  padding: 8px;
  max-height: 240px;
  overflow-y: auto;
  font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
}
.ltl-tok { cursor: pointer; border-radius: 3px; }
.ltl-tok:hover { background: #e8eefc; }
.ltl-tok.ltl-on { background: #ffe08a; }
.ltl-eos { opacity: 0.55; }
.ltl-log {
  height: 280px !important;
  max-height: 280px !important;
  overflow-y: auto !important;
  flex: 0 0 auto !important;
}
</style>
<script>
(function () {
  if (window.__ltlTokenClick) { return; }
  window.__ltlTokenClick = true;
  window.addEventListener("click", function (ev) {
    var span = ev.target && ev.target.closest && ev.target.closest(".ltl-tok");
    if (!span) { return; }
    var box = document.querySelector(".ltl-pick input");
    if (!box) { return; }
    window.__ltlPickN = (window.__ltlPickN || 0) + 1;
    var gen = span.getAttribute("data-gen") || "0";
    var i = span.getAttribute("data-i");
    box.value = window.__ltlPickN + "," + gen + "," + i;
    box.dispatchEvent(new Event("input", { bubbles: true }));
  }, true);
  if (!window.__ltlLogFollow) {
    window.__ltlLogFollow = true;
    setInterval(function () {
      var box = document.querySelector(".ltl-log");
      if (!box) { return; }
      var gap = box.scrollHeight - box.scrollTop - box.clientHeight;
      if (gap < 80) { box.scrollTop = box.scrollHeight; }
    }, 250);
  }
})();
</script>
"""
))
display(widgets.VBox([
    load_mode,
    widgets.HBox([gemma_dd, loc_dd]),
    widgets.HBox([full_dd, refresh_btn, load_btn, new_btn, clear_btn]),
    question,
    widgets.HBox([generate_btn, show_all, legend]),
    readout,
    widgets.HBox([
        widgets.VBox(
            [status, board],
            layout=widgets.Layout(flex="0 0 auto"),
        ),
        widgets.VBox(
            [settings.box],
            layout=widgets.Layout(flex="1 0 520px", min_width="520px"),
        ),
    ], layout=widgets.Layout(width="100%", align_items="flex-start")),
    token_html,
    log,
    pick_box,
]))
